# 13 CAGE-4 Baseline Rollouts

Run this notebook after the previous numbered stage completes.


In [1]:
from pathlib import Path
import os, sys

REPO_ROOT_OVERRIDE = None


def _looks_like_repo(base: Path) -> bool:
    return (base / "pyproject.toml").exists() and (base / "src" / "cybsage_marl").exists()


def _mount_drive_if_colab():
    if not Path("/content").exists():
        return
    if Path("/content/drive/MyDrive").exists() or Path("/content/drive/My Drive").exists():
        return
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive auto-mount failed:", exc)


def _find_repo_root():
    _mount_drive_if_colab()
    candidates = []
    if REPO_ROOT_OVERRIDE:
        candidates.append(Path(REPO_ROOT_OVERRIDE).expanduser())
    if os.environ.get("CYBSAGE_REPO_ROOT"):
        candidates.append(Path(os.environ["CYBSAGE_REPO_ROOT"]).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend([
        Path("/content/drive/MyDrive/projects/implementation/IMPLEMENTATION"),
        Path("/content/drive/My Drive/projects/implementation/IMPLEMENTATION"),
        Path("/content/drive/MyDrive/IMPLEMENTATION"),
        Path("/content/drive/My Drive/IMPLEMENTATION"),
        Path("/content/drive/MyDrive/projects/implementation/CybSage-MARL[DeepakSir]/IMPLEMENTATION"),
        Path("/content/drive/My Drive/projects/implementation/CybSage-MARL[DeepakSir]/IMPLEMENTATION"),
        Path("/Volumes/LocalDrive1/onlyscholar-projects/CybSage-MARL[DeepakSir]/IMPLEMENTATION"),
    ])

    checked = []
    seen = set()
    for base in candidates:
        base = base.expanduser()
        if str(base) in seen:
            continue
        seen.add(str(base))
        checked.append(base)
        if _looks_like_repo(base):
            return base

    # Last-resort bounded search. This catches Drive shortcuts/copies while avoiding a full Drive crawl.
    search_roots = [Path("/content/drive/MyDrive"), Path("/content/drive/My Drive"), Path("/content")]
    for root in search_roots:
        if not root.exists():
            continue
        try:
            for pyproject in root.glob("**/pyproject.toml"):
                base = pyproject.parent
                checked.append(base)
                if base.name == "IMPLEMENTATION" and _looks_like_repo(base):
                    return base
        except Exception as exc:
            checked.append(Path(f"{root} [search failed: {exc}]"))

    raise RuntimeError(
        "Could not find CybSAGE IMPLEMENTATION repo root visible to this runtime.\n\n"
        "Most common fix: in Colab, run drive.mount('/content/drive') and confirm the folder exists under "
        "/content/drive/MyDrive/projects/implementation/IMPLEMENTATION.\n\n"
        "Checked:\n" + "\n".join(map(str, checked[-80:]))
    )


REPO_ROOT = _find_repo_root()
os.chdir(REPO_ROOT)
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print("Repository root:", REPO_ROOT)


Repository root: /content/drive/MyDrive/projects/implementation/IMPLEMENTATION


In [2]:
import json
from pathlib import Path
summary_path = Path("outputs/12_cage4_dependency_check/dependency_summary.json")
if summary_path.exists():
    print(json.loads(summary_path.read_text()))
else:
    print("Run notebook 12 first.")

{'cyborg_repo': '/content/drive/MyDrive/projects/Dataset/Security/CybORG_CAGE_Challenge_4/cage-challenge-4', 'cyborg_repo_ok': True, 'python': '/usr/bin/python3', 'ready_for_cage4_rollouts': True, 'missing_required_packages': [], 'missing_optional_packages': ['ray', 'pettingzoo'], 'wrapper_import_ok': True, 'wrapper_import_error': '', 'ray_rllib_stub_used': True, 'install_hint': ''}


In [3]:
!PYTHONPATH=src python scripts/run_stage.py cage4_baselines --cage4-episodes 2 --cage4-max-steps 100 --cage4-seeds 42 43 44 45 46

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.13/dist-packages/pkg_resources/__init__.py:3148: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`

In [4]:
import pandas as pd
from pathlib import Path
path = Path("outputs/13_cage4_baseline_rollouts/cage4_baseline_summary.csv")
pd.read_csv(path) if path.exists() and path.stat().st_size else "No baseline summary produced; check cage4_baseline_summary.json"

,policy,episodes,reward_mean,reward_std,reward_min,reward_max,mean_step_reward,steps_mean,wall_time_sec_mean
0,decoy_priority,10,-57.3,31.244733,-92.0,-8.0,-0.578788,99.0,4.966576
1,remove_priority,10,-115.0,83.382652,-268.0,-20.0,-1.161616,99.0,4.122158
2,monitor,10,-133.7,100.805699,-312.0,-20.0,-1.350505,99.0,4.222734
3,sleep,10,-133.7,100.805699,-312.0,-20.0,-1.350505,99.0,4.143631
4,random,10,-456.9,162.035970,-724.0,-227.0,-4.615152,99.0,4.316201
5,restore_priority,10,-588.2,79.170421,-728.0,-517.0,-5.941414,99.0,4.226192
